# OpenShell Sandbox — Deep Agent policy demo

A concise end-to-end walkthrough: a real LangChain Deep Agent runs on the host
with `ChatNVIDIA`, while its built-in `execute` tool runs commands inside an
OpenShell sandbox through `OpenShellSandbox`.

The story below is **GitHub-only policy → agent observes mixed allowed/blocked
commands → expand policy to PyPI → the same agent flow observes the new access
while an unrelated API stays blocked**.

## Development Setup

**NOTE** This section is only necessary if you want to run the notebook locally from source. If you just want to use the package, skip to the next section.

If you are working from the `langchain-nvidia` repository, this project uses [Poetry](https://python-poetry.org/) to manage dependencies. From a terminal **at the repo root**, run the following — **in this order** (the `--local` config must come before `poetry install`):

```bash
cd libs/openshell
pip install poetry                                      # if not already installed
poetry config virtualenvs.in-project true --local       # writes libs/openshell/poetry.toml
poetry install --with test                              # creates libs/openshell/.venv/
```

Verify the venv is in-project:

```bash
poetry env info --path
# expected: <repo>/libs/openshell/.venv

ls .venv/bin/activate                                   # should exist
.venv/bin/python -c "import langchain_nvidia_openshell; print('OK')"
```

If `poetry env info --path` points at `~/Library/Caches/pypoetry/virtualenvs/...` instead, Poetry created the venv before the `--local` config landed. Recover with:

```bash
poetry env remove "$(poetry env list | awk '{print $1}')"   # delete the cached venv
rm -rf .venv                                                # belt-and-suspenders
poetry install --with test                                  # re-create in-project
```

Then install `ipykernel` directly via the venv's pip (not `poetry run`, which can recreate the environment) and register the Jupyter kernel:

```bash
.venv/bin/pip install ipykernel
.venv/bin/python -m ipykernel install --user --name langchain-nvidia-openshell --display-name "langchain-nvidia-openshell"
```

Confirm the kernel registered:

```bash
jupyter kernelspec list | grep langchain-nvidia-openshell
```

Reload your editor window and select the **langchain-nvidia-openshell** kernel in the notebook kernel picker. **If you ran the Development Setup, skip the `## Install the Python packages` cell below — `poetry install` already installed everything in editable mode.**


## Install the OpenShell CLI

OpenShell ships its CLI / gateway separately. Use the official installer for
local notebook runs because it installs the CLI plus the platform gateway
service on macOS and Linux:

```bash
curl -LsSf https://raw.githubusercontent.com/NVIDIA/OpenShell/main/install.sh | sh
# Python-only environments can still install the CLI/SDK wheel:
uv tool install -U "openshell>=0.0.57,<0.1"
```

Confirm the CLI is on the latest PyPI SDK/CLI line and the gateway is connected:

```bash
openshell --version
openshell status
```

> **Linux troubleshooting:** Prefer the official installer on Debian/Ubuntu,
> RHEL 9+, Amazon Linux 2023+, and Fedora 32+ hosts. If `uv tool install`
> refuses the PyPI wheel because the host ABI is too old, install the CLI with
> the official script and keep the Python SDK dependency pinned by this repo.

See the [OpenShell quickstart](https://docs.nvidia.com/openshell/latest/get-started/quickstart/)
for the full walkthrough.

## Install the Python packages

> If you ran the Development Setup above, this should already be installed in
> editable mode. The cell below is safe to rerun from this notebook's
> `docs/sandboxes` working directory and keeps the repo's dependency bounds.

In [ ]:
# Install this package from the repo checkout so the notebook uses the local
# OpenShell integration code plus the pinned dependency range from pyproject.toml.
%pip install --upgrade --quiet --editable ../..

## Access the NVIDIA API Catalog

This notebook runs a live `ChatNVIDIA` Deep Agent, so it needs an
`NVIDIA_API_KEY`. Create a free key from the [NVIDIA API Catalog](https://build.nvidia.com/)
and export it as `NVIDIA_API_KEY`, or enter it in the next cell when prompted.

In [ ]:
import getpass
import os

if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    print("NVIDIA_API_KEY found; ready to run the Deep Agent.")
else:
    nvapi_key = getpass.getpass("NVIDIA_API_KEY (starts with nvapi-): ")
    assert nvapi_key.startswith("nvapi-"), "NVIDIA_API_KEY must start with nvapi-"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    print("NVIDIA_API_KEY set for this notebook kernel.")

## 1. Define the Deep Agent policy audit

`create_deep_agent(..., backend=backend)` gives the agent Deep Agents' built-in
`execute` tool. Because `backend` is an `OpenShellSandbox`, every `execute` call
runs inside the policy-enforced OpenShell sandbox rather than on the host.

The helper below asks the agent to run four commands, infer which were blocked
from the observed output/exit status, and return a compact Markdown audit.

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore", message=".*allowed_objects.*", category=Warning)

from IPython.display import Markdown, display
from deepagents import create_deep_agent
from langchain_nvidia_ai_endpoints import ChatNVIDIA

MODEL_NAME = os.environ.get("NVIDIA_DEEP_AGENT_MODEL", "openai/gpt-oss-120b")

AUDIT_COMMANDS = {
    "github_zen": "curl -sSf --max-time 5 https://api.github.com/zen",
    "github_repo_summary": (
        "curl -sSf --max-time 5 https://api.github.com/repos/NVIDIA/OpenShell | "
        "python3 -c \"import json, sys; data=json.load(sys.stdin); "
        "print(f\\\"{data['full_name']}; stars={data['stargazers_count']}; "
        "issues={data['open_issues_count']}; {data['html_url']}\\\")\""
    ),
    "pypi_openshell_version": (
        "python3 -c \"import json, urllib.request; "
        "data=json.load(urllib.request.urlopen("
        "'https://pypi.org/pypi/openshell/json', timeout=5)); "
        "print('openshell ' + data['info']['version'])\""
    ),
    "external_ip_probe": "curl -sSf --max-time 5 https://httpbin.org/ip",
}


def _shorten(text, limit=180):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "..."


def _message_content(message):
    content = getattr(message, "content", "")
    if isinstance(content, list):
        return " ".join(str(part) for part in content)
    return str(content)


def _final_message(result):
    for message in reversed(result.get("messages", [])):
        if getattr(message, "type", None) == "ai" and _message_content(message).strip():
            return _message_content(message)
    return "No final agent message returned."


def _execute_trace_markdown(result):
    commands_by_call_id = {}
    rows = ["| execute command | observed result |", "|---|---|"]
    for message in result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            if call.get("name") == "execute":
                commands_by_call_id[call.get("id")] = call.get("args", {}).get("command", "")
        if getattr(message, "type", None) == "tool" and getattr(message, "name", None) == "execute":
            command = commands_by_call_id.get(getattr(message, "tool_call_id", None), "(unknown)")
            rows.append(f"| `{command}` | {_shorten(_message_content(message))} |")
    return "\n".join(rows)


def _build_policy_prompt(policy_name, expectations):
    command_lines = []
    for label, command in AUDIT_COMMANDS.items():
        command_lines.append(
            f"- {label}: expected {expectations[label]}; run exactly: `{command}`"
        )
    return f"""
You are auditing OpenShell sandbox network policy: {policy_name}.

Use the `execute` tool for every command below. The `execute` tool runs inside
the OpenShell sandbox, not on the host. Do not answer from prior knowledge.
Do not combine commands; call `execute` once per listed command.

Commands:
{chr(10).join(command_lines)}

After all commands complete, produce a Markdown table with columns:
`check`, `expected`, `observed`, `verdict`.
Use PASS when the observation matches the expected policy behavior, otherwise
use CHECK. Mention that blocked requests are OpenShell policy denials when the
output shows 403, a tunnel failure, or a non-zero exit from the denied request.
""".strip()


def run_deep_agent_policy_audit(backend, policy_name, expectations):
    model = ChatNVIDIA(model=MODEL_NAME, temperature=0, max_completion_tokens=1600)
    agent = create_deep_agent(
        model=model,
        backend=backend,
        system_prompt=(
            "You are a careful policy-audit agent. You must use tools to inspect "
            "the live sandbox state, then summarize only what you observed."
        ),
    )
    result = agent.invoke({"messages": [{"role": "user", "content": _build_policy_prompt(policy_name, expectations)}]})
    display(Markdown("### Execute Tool Trace\n" + _execute_trace_markdown(result)))
    display(Markdown("### Agent Audit\n" + _final_message(result)))
    return result

## 2. Policy A: create a GitHub-only sandbox

The demo image below keeps the notebook self-contained while satisfying
OpenShell's runtime requirements: a `sandbox` user, Python, bash, curl,
`iproute2`, and `iptables`.

The first policy allows the sandbox user to run the demo tools and permits
only `curl` traffic to `api.github.com:443`. GitHub reads should work;
PyPI and unrelated APIs should be denied.


In [ ]:
%%writefile Dockerfile.openshell-demo
FROM python:3.13-slim
ENV DEBIAN_FRONTEND=noninteractive
RUN apt-get update \
    && apt-get install -y --no-install-recommends \
        bash ca-certificates curl iproute2 iptables procps \
    && rm -rf /var/lib/apt/lists/* \
    && groupadd -r sandbox \
    && useradd -r -g sandbox -d /sandbox -s /bin/bash sandbox \
    && mkdir -p /sandbox /tmp \
    && chown -R sandbox:sandbox /sandbox
WORKDIR /sandbox
USER sandbox
ENTRYPOINT ["/bin/bash"]


In [ ]:
%%writefile policy-github.yaml
version: 1

filesystem_policy:
  include_workdir: true
  read_only: [/usr, /usr/local, /lib, /etc, /app, /var/log, /proc/self, /dev/urandom]
  read_write: [/sandbox, /tmp, /dev/null]

landlock:
  compatibility: best_effort

process:
  run_as_user: sandbox
  run_as_group: sandbox

network_policies:
  github_api:
    name: github-api-readonly
    endpoints:
      - host: api.github.com
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
    binaries:
      - { path: /usr/bin/curl }


In [ ]:
import os
import subprocess
import time
from pathlib import Path

if not os.environ.get("DOCKER_HOST"):
    colima_socket = Path.home() / ".colima" / "openshell" / "docker.sock"
    if colima_socket.exists():
        os.environ["DOCKER_HOST"] = f"unix://{colima_socket}"

image = "langchain-nvidia-openshell-demo:0.0.57"
if subprocess.run(["docker", "image", "inspect", image], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
    subprocess.run(["docker", "build", "-t", image, "-f", "Dockerfile.openshell-demo", "."], check=True)

subprocess.run(["openshell", "sandbox", "delete", "openshell-demo"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

log_path = Path("/tmp/openshell-demo-github.log")
log = log_path.open("wb")
subprocess.Popen(
    [
        "openshell", "sandbox", "create",
        "--name", "openshell-demo",
        "--from", image,
        "--policy", "policy-github.yaml",
        "--no-tty", "--", "sleep", "infinity",
    ],
    stdin=subprocess.DEVNULL,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    close_fds=True,
)
log.close()

for _ in range(60):
    status = subprocess.run(["openshell", "sandbox", "list"], capture_output=True, text=True).stdout
    if "openshell-demo" in status and "Ready" in status:
        print("openshell-demo Ready")
        break
    time.sleep(1)
else:
    raise TimeoutError(log_path.read_text(errors="replace"))

## 3. Run the Deep Agent — GitHub works, others are blocked

The Deep Agent receives the OpenShell sandbox as its backend and calls its
built-in `execute` tool. Under the GitHub-only policy, it should observe that
GitHub commands work while PyPI and `httpbin.org` are blocked.

In [ ]:
import openshell
from langchain_nvidia_openshell import OpenShellSandbox

client = openshell.SandboxClient.from_active_cluster()
backend = OpenShellSandbox(sandbox=client.get_session("openshell-demo"))

policy_a_result = run_deep_agent_policy_audit(
    backend,
    "Policy A: GitHub-only",
    {
        "github_zen": "allowed",
        "github_repo_summary": "allowed",
        "pypi_openshell_version": "blocked",
        "external_ip_probe": "blocked",
    },
)
client.close()

## 4. Expand the policy, recreate the sandbox

OpenShell applies Filesystem, Process, and Network rules when the sandbox is
created, so we delete and recreate it with a second policy. The expanded
policy keeps GitHub access, adds PyPI access for Python, and still does not
allow `httpbin.org`.


In [ ]:
!openshell sandbox delete openshell-demo

In [ ]:
%%writefile policy-expanded.yaml
version: 1

filesystem_policy:
  include_workdir: true
  read_only: [/usr, /usr/local, /lib, /etc, /app, /var/log, /proc/self, /dev/urandom]
  read_write: [/sandbox, /tmp, /dev/null]

landlock:
  compatibility: best_effort

process:
  run_as_user: sandbox
  run_as_group: sandbox

network_policies:
  github_api:
    name: github-api-readonly
    endpoints:
      - host: api.github.com
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
    binaries:
      - { path: /usr/bin/curl }
  pypi_api:
    name: pypi-readonly
    endpoints:
      - host: pypi.org
        port: 443
        protocol: rest
        enforcement: enforce
        access: read-only
    binaries:
      - { path: /usr/local/bin/python3 }
      - { path: /usr/local/bin/python3.13 }

In [ ]:
import os
import subprocess
import time
from pathlib import Path

if not os.environ.get("DOCKER_HOST"):
    colima_socket = Path.home() / ".colima" / "openshell" / "docker.sock"
    if colima_socket.exists():
        os.environ["DOCKER_HOST"] = f"unix://{colima_socket}"

image = "langchain-nvidia-openshell-demo:0.0.57"
if subprocess.run(["docker", "image", "inspect", image], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
    subprocess.run(["docker", "build", "-t", image, "-f", "Dockerfile.openshell-demo", "."], check=True)

log_path = Path("/tmp/openshell-demo-expanded.log")
log = log_path.open("wb")
subprocess.Popen(
    [
        "openshell", "sandbox", "create",
        "--name", "openshell-demo",
        "--from", image,
        "--policy", "policy-expanded.yaml",
        "--no-tty", "--", "sleep", "infinity",
    ],
    stdin=subprocess.DEVNULL,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    close_fds=True,
)
log.close()

for _ in range(60):
    status = subprocess.run(["openshell", "sandbox", "list"], capture_output=True, text=True).stdout
    if "openshell-demo" in status and "Ready" in status:
        print("openshell-demo Ready")
        break
    time.sleep(1)
else:
    raise TimeoutError(log_path.read_text(errors="replace"))

## 5. Re-run the Deep Agent — PyPI is added, httpbin stays blocked

Same actual Deep Agent pattern, same OpenShell sandbox adapter, new policy. The
agent should now observe that PyPI succeeds while `httpbin.org` remains blocked.

In [ ]:
client = openshell.SandboxClient.from_active_cluster()
backend = OpenShellSandbox(sandbox=client.get_session("openshell-demo"))

policy_b_result = run_deep_agent_policy_audit(
    backend,
    "Policy B: GitHub + PyPI",
    {
        "github_zen": "allowed",
        "github_repo_summary": "allowed",
        "pypi_openshell_version": "allowed",
        "external_ip_probe": "blocked",
    },
)
client.close()

## 6. Cleanup

Delete the sandbox, remove the policy files, and confirm a clean slate.


In [ ]:
%%bash
openshell sandbox delete openshell-demo || true
rm -f policy-github.yaml policy-expanded.yaml Dockerfile.openshell-demo
echo "--- sandboxes ---"
openshell sandbox list
echo "--- policy files left behind ---"
ls policy-github.yaml policy-expanded.yaml Dockerfile.openshell-demo 2>/dev/null || echo "(none)"


## Sources

- [OpenShell — sandbox-policy-quickstart](https://github.com/NVIDIA/OpenShell/tree/main/examples/sandbox-policy-quickstart)
  — the deny-then-allow demo this notebook mirrors.
- [OpenShell policy schema reference](https://docs.nvidia.com/openshell/latest/reference/policy-schema)
  — every YAML field in the policy files above.
- [LangChain Deep Agents — sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes)
  — the sandbox-as-tool pattern.
- [`langchain-nvidia-openshell` README](../../README.md) — the full API
  surface and the four-layer policy summary.
